# ベースライン学習（最高精度パイプライン・検証: 時系列CV）

- **このノートでやること**: **現在の最高精度パイプライン**（55 特徴 + BPR 64 の 128 列 ＝ 計 183 特徴）で時系列 CV と提出 1 本。Public **0.76479** を出すのと同じ構成。
- **使う特徴量**: 55 特徴（embedding + PCA16 + doc_x_critic_te）＋ BPR 64 の批評家・映画ベクトル 128 列。`get_setup(use_best_pipeline=True)` と `get_bpr_base(ctx, factors=64)` で取得。
- **検証**: 時系列CV（2013〜2016 を val）。提出は全 train で 1 本学習して test を予測。
- **本番ベースライン（超える目標）**: Kaggle Public **0.76479**。改善実験はこのスコアを下回らないようにする。伸びそうな改善は `docs/09_IMPROVEMENT_NEXT.md` を参照。

In [1]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

from lib import BASELINE_LGB_PARAMS, add_prediction_analysis, summarize_errors_by, run_full_analysis
from lib.improvement_candidates import get_setup
from lib.improvement_candidates import get_bpr_base
from lib.submission import save_submission, verify_submission

OUTPUT_DIR = "outputs"
SUBMISSION_FILENAME = "submission_2hop_bpr64_only.csv"  # 提出は常にこの1ファイルに上書き・検証まで自動
os.makedirs(os.path.join(OUTPUT_DIR, "submissions"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "analysis"), exist_ok=True)
print("Setup complete.")

Setup complete.


In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

seed_everything(42)

In [3]:
# 最高精度パイプライン: 55 特徴 + BPR 64（128 列）= 計 183 特徴
ctx = get_setup(seed=42, output_dir=OUTPUT_DIR, use_best_pipeline=True)
train_df, test_df, feats = get_bpr_base(ctx, factors=64)
# 非数値・非カテゴリ列を category に（LGB 用）
for col in feats:
    if not pd.api.types.is_numeric_dtype(train_df[col]) and not pd.api.types.is_categorical_dtype(train_df[col]):
        train_df[col] = train_df[col].astype("category")
        test_df[col] = test_df[col].astype("category")
train = train_df
test = test_df
FEATURES = feats
y = ctx.y
time_splits = ctx.time_splits
VAL_YEARS = [2013, 2014, 2015, 2016]
print(f"Train: {len(train):,}, Test: {len(test):,}, Features: {len(FEATURES)}")

  0%|          | 0/100 [00:00<?, ?it/s]

Train: 653,507, Test: 40,716, Features: 183


In [4]:
train.head()

,ID,rotten_tomatoes_link,critic_name,top_critic,publisher_name,review_date,movie_title,movie_info,content_rating,genres,...,implicit_bpr_64_i_54,implicit_bpr_64_i_55,implicit_bpr_64_i_56,implicit_bpr_64_i_57,implicit_bpr_64_i_58,implicit_bpr_64_i_59,implicit_bpr_64_i_60,implicit_bpr_64_i_61,implicit_bpr_64_i_62,implicit_bpr_64_i_63
0,0,m/0814255,Andrew L. Urban,False,Urban Cinefile,2010-02-06,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,-0.518441,-0.394004,0.244693,0.061099,-0.201449,0.086679,0.004488,0.143388,-0.400918,-0.043982
1,1,m/0814255,Louise Keller,False,Urban Cinefile,2010-02-06,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,-0.518441,-0.394004,0.244693,0.061099,-0.201449,0.086679,0.004488,0.143388,-0.400918,-0.043982
2,2,m/0814255,David Germain,True,Associated Press,2010-02-10,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,-0.518441,-0.394004,0.244693,0.061099,-0.201449,0.086679,0.004488,0.143388,-0.400918,-0.043982
3,3,m/0814255,Nick Schager,False,Slant Magazine,2010-02-10,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,-0.518441,-0.394004,0.244693,0.061099,-0.201449,0.086679,0.004488,0.143388,-0.400918,-0.043982
4,4,m/0814255,Bill Goodykoontz,True,Arizona Republic,2010-02-10,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,-0.518441,-0.394004,0.244693,0.061099,-0.201449,0.086679,0.004488,0.143388,-0.400918,-0.043982


In [5]:
# time_splits と VAL_YEARS はデータセルで ctx から設定済み
print(f"時系列CV: {len(time_splits)} folds (val years = {VAL_YEARS})")
for i, (_, val_idx) in enumerate(time_splits):
    print(f"  Fold{i+1}: val_year={VAL_YEARS[i]}, val_n={len(val_idx):,}")

時系列CV: 4 folds (val years = [2013, 2014, 2015, 2016])
  Fold1: val_year=2013, val_n=47,263
  Fold2: val_year=2014, val_n=45,165
  Fold3: val_year=2015, val_n=49,842
  Fold4: val_year=2016, val_n=36,455


In [6]:
X = train[FEATURES]
X_test = test[FEATURES]
# y, time_splits はデータセルで ctx から設定済み

lgb_params = ctx.lgb_params

oof = np.zeros(len(train))
test_preds = []
fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(time_splits):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(30, verbose=False)])
    oof[val_idx] = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, oof[val_idx])
    fold_scores.append(fold_auc)
    test_preds.append(model.predict_proba(X_test)[:, 1])
    print(f"Fold{fold+1}: val_year={VAL_YEARS[fold]}, AUC={fold_auc:.4f}")

# 時系列CVでは oof が全行を埋めないため、CV AUC は fold 平均で表示
print(f"\nCV AUC (fold mean): {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

Fold1: val_year=2013, AUC=0.7821
Fold2: val_year=2014, AUC=0.7918
Fold3: val_year=2015, AUC=0.7865
Fold4: val_year=2016, AUC=0.7813

CV AUC (fold mean): 0.7855 ± 0.0042


In [7]:
# 提出用: 全 train で 1 本学習して test を予測（ベースラインと同じく「全データで学習→予測」）
model_full = lgb.LGBMClassifier(**lgb_params)
model_full.fit(X, y)
final_pred = model_full.predict_proba(X_test)[:, 1]

out_path = ctx.submissions_dir / SUBMISSION_FILENAME
save_submission(test, final_pred, out_path, sanitize=True)
ver = verify_submission(out_path, test)
if ver["ok"]:
    print(f"提出ファイル: {ver['path']} (rows={ver['rows']:,}) — 検証 OK")
else:
    print(f"提出ファイル: {ver['path']} — 検証 NG: {ver['message']}")

# 特徴量重要度（model_full の gain）
importance_df = pd.DataFrame({
    "feature": FEATURES,
    "importance": model_full.feature_importances_,
}).sort_values("importance", ascending=False)
display(importance_df)
print(f"\n重要度 Top5: {list(importance_df.head(5)['feature'].values)}")

提出ファイル: outputs/submissions/submission_2hop_bpr64_only.csv (rows=40,716) — 検証 OK


,feature,importance
3,publisher_name,5960
7,directors,5460
1,critic_name,4170
0,rotten_tomatoes_link,3514
8,authors,3131
...,...,...
25,genre_Horror,7
31,runtime_bin,5
23,genre_Fantasy,3
24,genre_Romance,3



重要度 Top5: ['publisher_name', 'directors', 'critic_name', 'rotten_tomatoes_link', 'authors']


In [8]:
# 予測の当たり・外れ分析（CV の oof を使い、どのセグメントで外しているかを集計）
# oof は時系列CVで val 部分のみ埋まるので、val に含まれる行だけ分析対象にする
val_mask = np.zeros(len(train), dtype=bool)
for _tr_idx, val_idx in time_splits:
    val_mask[val_idx] = True
train_val = train.loc[val_mask].copy()
oof_val = oof[val_mask]
y_val = train["target"].values[val_mask]
analysis_df, summaries = run_full_analysis(
    train_val, y_val, oof_val,
    group_cols=["review_year", "content_rating", "genre_Drama", "genre_Documentary", "top_critic"],
    min_count=100,
)
# 分析結果の保存（改善のヒント用）
analysis_df.to_csv(os.path.join(OUTPUT_DIR, "analysis", "train_with_predictions.csv"), index=False)
print(f"Saved {OUTPUT_DIR}/analysis/train_with_predictions.csv (target, pred, outcome=TP/TN/FP/FN 付き)")

for col, summary in summaries.items():
    fname = os.path.join(OUTPUT_DIR, "analysis", f"prediction_summary_by_{col}.csv")
    summary.to_csv(fname, index=False)
    print(f"  {fname}: n={len(summary)}")
    # AUC が低い or FN/FP 率が高いセグメントを表示
    if "auc" in summary.columns and summary["auc"].notna().any():
        worst = summary.loc[summary["auc"].idxmin()]
        print(f"    最低AUC: {worst[col]} (AUC={worst['auc']:.4f}, n={worst['n']})")
print("\nセグメント別集計の例 (review_year):")
display(summaries.get("review_year", pd.DataFrame()))

Saved outputs/analysis/train_with_predictions.csv (target, pred, outcome=TP/TN/FP/FN 付き)
  outputs/analysis/prediction_summary_by_review_year.csv: n=4
    最低AUC: 2016.0 (AUC=0.7813, n=36455.0)
  outputs/analysis/prediction_summary_by_content_rating.csv: n=6
    最低AUC: NC17 (AUC=0.7027, n=240)
  outputs/analysis/prediction_summary_by_genre_Drama.csv: n=2
    最低AUC: 0.0 (AUC=0.7822, n=84473.0)
  outputs/analysis/prediction_summary_by_genre_Documentary.csv: n=2
    最低AUC: 1.0 (AUC=0.7473, n=14931.0)
  outputs/analysis/prediction_summary_by_top_critic.csv: n=2
    最低AUC: True (AUC=0.7759, n=54956)

セグメント別集計の例 (review_year):


,review_year,n,n_TP,n_TN,n_FP,n_FN,accuracy,auc,FP_rate,FN_rate,pos_rate,pred_pos_rate
0,2013,47263,26410,8400,8319,4134,0.736517,0.782123,0.497578,0.135346,0.646256,0.734803
1,2014,45165,25532,7889,8303,3441,0.739976,0.791842,0.512784,0.118766,0.641492,0.749142
2,2015,49842,28568,8289,8534,4451,0.739477,0.786504,0.507282,0.134801,0.662473,0.744392
3,2016,36455,20877,5984,6397,3197,0.736826,0.781341,0.516679,0.132799,0.660376,0.748155
